# 5.7 Gradient Descent Optimization

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 5: Linear Regression

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-05-linear-regression/05.07_gradient-descent-optimization.ipynb)

Companion code for Section 5.7 of the book. Run the cells in order from top to bottom.

**Example 5.7.1**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

np.random.seed(42)
X, y = make_regression(n_samples=1000, n_features=3, noise=20, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_tr = scaler.fit_transform(X_train)
X_te = scaler.transform(X_test)

def mse(y_true, y_pred): return np.mean((y_true - y_pred)**2)
def predict(X, w, b):    return X @ w + b

# ── Batch Gradient Descent ────────────────────────────────────────
def batch_gd(X, y, lr=0.05, n_epochs=200):
    n, p = X.shape
    w, b = np.zeros(p), 0.0
    history = []
    for _ in range(n_epochs):
        errors = predict(X, w, b) - y
        dw = (1/n) * X.T @ errors
        db = (1/n) * errors.sum()
        w -= lr * dw
        b -= lr * db
        history.append(mse(y, predict(X, w, b)))
    return w, b, history

# ── Stochastic Gradient Descent ───────────────────────────────────
def sgd(X, y, lr=0.01, n_epochs=200):
    n, p = X.shape
    w, b = np.zeros(p), 0.0
    history = []
    for epoch in range(n_epochs):
        idx = np.random.permutation(n)
        for i in idx:
            xi, yi = X[i], y[i]
            error  = predict(xi, w, b) - yi
            w -= lr * xi * error
            b -= lr * error
        history.append(mse(y, predict(X, w, b)))
    return w, b, history

# ── Mini-Batch Gradient Descent ───────────────────────────────────
def mini_batch_gd(X, y, lr=0.05, batch_size=32, n_epochs=200):
    n, p = X.shape
    w, b = np.zeros(p), 0.0
    history = []
    for epoch in range(n_epochs):
        idx = np.random.permutation(n)
        X_shuf, y_shuf = X[idx], y[idx]
        for start in range(0, n, batch_size):
            Xb = X_shuf[start:start+batch_size]
            yb = y_shuf[start:start+batch_size]
            errors = predict(Xb, w, b) - yb
            w -= lr * (1/len(Xb)) * Xb.T @ errors
            b -= lr * (1/len(Xb)) * errors.sum()
        history.append(mse(y, predict(X, w, b)))
    return w, b, history

# Run all three
w_bgd, b_bgd, h_bgd = batch_gd(X_tr, y_train)
w_sgd, b_sgd, h_sgd = sgd(X_tr, y_train)
w_mbgd,b_mbgd,h_mbgd= mini_batch_gd(X_tr, y_train, batch_size=32)

print(f'{'Method':<20} {'Final MSE':>12} {'Test MSE':>12}')
print('-'*46)
for name, w, b, h in [('Batch GD',   w_bgd, b_bgd, h_bgd),
                       ('SGD',         w_sgd, b_sgd, h_sgd),
                       ('Mini-Batch(32)',w_mbgd,b_mbgd,h_mbgd)]:
    train_mse = h[-1]
    test_mse  = mse(y_test, predict(X_te, w, b))
    print(f'{name:<20} {train_mse:>12.4f} {test_mse:>12.4f}')

# ── Learning Rate Comparison ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: convergence curves
ax = axes[0]
epochs = range(1, 201)
ax.plot(epochs, h_bgd,  'b-', lw=2,  label='Batch GD (lr=0.05)',    alpha=0.9)
ax.plot(epochs, h_sgd,  'r-', lw=1.5,label='SGD (lr=0.01)',         alpha=0.7)
ax.plot(epochs, h_mbgd, 'g-', lw=2,  label='Mini-Batch/32 (lr=0.05)',alpha=0.9)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('MSE Loss', fontsize=12)
ax.set_title('Convergence Comparison', fontsize=13, fontweight='bold')
ax.set_yscale('log'); ax.legend(); ax.grid(alpha=0.3)

# Right: learning rate effect on Batch GD
ax = axes[1]
for lr, color, ls in [(0.001,'purple',':'), (0.01,'blue','--'),
                       (0.1,'green','-'), (0.5,'orange','-'), (1.0,'red','-')]:
    try:
        _, _, h = batch_gd(X_tr, y_train, lr=lr, n_epochs=100)
        if h[-1] < 1e8:
            ax.plot(h, color=color, ls=ls, lw=2, label=f'lr={lr}')
        else:
            ax.plot([h[0]], [h[0]], color=color, marker='x', ms=15, label=f'lr={lr} (diverged)')
    except: pass
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('MSE Loss', fontsize=12)
ax.set_title('Learning Rate Sensitivity', fontsize=13, fontweight='bold')
ax.set_yscale('log'); ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('gradient_descent_comparison.png', dpi=150, bbox_inches='tight')
plt.show()